In [ ]:
from pathlib import Path
import re
import warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

In [ ]:
warnings.filterwarnings("ignore")

=========================================================
1. 文件路径：只需要修改这里
=========================================================

In [ ]:
CSV_PATH = Path(
    r"C:\Users\lichu\PycharmProjects\PythonProject3\3CE问卷数据-2026-08-12.csv"
)

In [ ]:
# 分析结果保存到CSV所在文件夹
OUTPUT_DIR = CSV_PATH.parent / "3CE问卷分析结果"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
EXCEL_OUTPUT = OUTPUT_DIR / "3CE消费者路径断点分析.xlsx"

=========================================================
2. 中文字体与图表样式
=========================================================

In [ ]:
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS"
]

In [ ]:
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
sns.set_theme(style="whitegrid")
sns.set_context("notebook")

=========================================================
3. 读取CSV，自动尝试常见编码
=========================================================

In [ ]:
def read_csv_auto(path):
    if not path.exists():
        raise FileNotFoundError(
            f"\n没有找到CSV文件：\n{path}\n"
            f"请检查CSV_PATH是否正确。"
        )

    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
        "gbk"
    ]

    last_error = None

    for encoding in encodings:
        try:
            data = pd.read_csv(path, encoding=encoding)
            print(f"CSV读取成功，使用编码：{encoding}")
            return data
        except UnicodeDecodeError as error:
            last_error = error

    raise UnicodeDecodeError(
        last_error.encoding,
        last_error.object,
        last_error.start,
        last_error.end,
        "无法识别CSV编码"
    )

In [ ]:
df = read_csv_auto(CSV_PATH)

In [ ]:
# 清理列名中的换行、重复空格和首尾空格
df.columns = (
    df.columns
    .astype(str)
    .str.replace("\r", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
print(f"数据规模：{df.shape[0]}行，{df.shape[1]}列")

=========================================================
4. 根据真实CSV设置列名
=========================================================

In [ ]:
COL = {
    "id": "记录ID",
    "submit_time": "提交时间",
    "age": "年龄",
    "city": "所在城市级别",

    "platform": "接触美妆内容的平台",
    "reason": "购买彩妆的主要原因",
    "difficulty": "购买彩妆的困难",
    "mismatch": "种草后发现不适合自己的频率",
    "brand_need": "最希望美妆品牌解决的问题",

    "q9": "3CE了解程度",

    "q10a": "Q10A｜持续购买3CE的原因",
    "q11a": "Q11A｜提高购买频率的方式",

    "q10b": "Q10B｜很少或未继续购买的原因",
    "q11b": "Q11B｜重新关注或购买的方式",

    "q10c": "Q10C｜知道但未购买的原因",
    "q11c": "Q11C｜提高首次购买可能性的体验",

    # CSV题号与原问卷含义疑似相反，这里按照CSV实际列名读取
    "usual_brand": "Q10D｜通常使用的彩妆品牌",
    "new_brand_reason": "Q11D｜关注新彩妆品牌的原因",

    "kdrama": "韩系影视内容关注度",
    "kdrama_element": "影响兴趣的韩剧元素",
    "role_identification": "影视角色代入感",
    "style": "3CE韩剧女主妆风格",
    "brand_info": "希望品牌了解的信息",

    "value": "化妆的最大价值",

    "q16a": "体验评分：韩剧角色测试",
    "q16b": "体验评分：场景妆容助手",
    "q16c": "体验评分：AI个人风格探索",
    "q16d": "体验评分：3CE女性圈层社区",
    "q16e": "体验评分：韩系潮流实验室",

    "open_text": "Q18品牌长期陪伴期待"
}

In [ ]:
# 检查分析所需列是否存在
required_columns = [
    COL["age"],
    COL["city"],
    COL["platform"],
    COL["reason"],
    COL["difficulty"],
    COL["mismatch"],
    COL["brand_need"],
    COL["q9"],
    COL["kdrama"],
    COL["q16a"],
    COL["q16b"],
    COL["q16c"],
    COL["q16d"],
    COL["q16e"],
    COL["value"]
]

In [ ]:
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

In [ ]:
if missing_columns:
    print("\n以下必要列在CSV中不存在：")

    for column in missing_columns:
        print("-", repr(column))

    print("\nCSV中的真实列名：")

    for i, column in enumerate(df.columns):
        print(i, repr(column))

    raise KeyError("列名未完全匹配，请检查COL字典。")

In [ ]:
print("必要分析列全部匹配成功。")

=========================================================
5. 数据清洗
=========================================================

In [ ]:
# 清理所有文本列
for column in df.columns:
    if df[column].dtype == "object":
        df[column] = (
            df[column]
            .astype(str)
            .str.replace("\r", " ", regex=False)
            .str.replace("\n", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        df[column] = df[column].replace({
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        })

In [ ]:
def normalize_dash(series):
    """
    将不同类型的横线统一成普通减号。
    例如：买过1–2次 → 买过1-2次
    """
    return (
        series
        .astype("string")
        .str.replace("－", "-", regex=False)
        .str.replace("–", "-", regex=False)
        .str.replace("—", "-", regex=False)
        .str.replace("−", "-", regex=False)
        .str.strip()
    )

In [ ]:
q9 = normalize_dash(df[COL["q9"]])

In [ ]:
print("\nQ9实际选项人数：")
print(q9.value_counts(dropna=False))

=========================================================
6. 构造消费者路径变量
=========================================================

In [ ]:
# 认知：不是“完全不了解”
df["品牌认知"] = q9.ne("完全不了解").astype(int)

In [ ]:
# 购买：经常购买或买过1-2次
df["购买过3CE"] = q9.isin([
    "经常购买",
    "买过1-2次"
]).astype(int)

In [ ]:
# 留存代理指标：经常购买
# 注意：因为问卷没有最近购买时间与复购次数，
# 所以这里只能称为“经常购买率”或“留存代理指标”
df["经常购买3CE"] = q9.eq("经常购买").astype(int)

In [ ]:
# 分配消费者路径阶段
conditions = [
    q9.eq("完全不了解"),
    q9.eq("听说过但没有购买"),
    q9.eq("买过1-2次"),
    q9.eq("经常购买")
]

In [ ]:
stage_labels = [
    "未认知",
    "已认知未购买",
    "购买但未形成稳定复购",
    "经常购买"
]

In [ ]:
df["消费者阶段"] = np.select(
    conditions,
    stage_labels,
    default="其他/需检查"
)

=========================================================
7. 构造升级兴趣指标
=========================================================

In [ ]:
Q16_COLS = [
    COL["q16a"],
    COL["q16b"],
    COL["q16c"],
    COL["q16d"],
    COL["q16e"]
]

In [ ]:
Q16_NAMES = {
    COL["q16a"]: "韩剧角色测试",
    COL["q16b"]: "场景妆容助手",
    COL["q16c"]: "AI个人风格探索",
    COL["q16d"]: "3CE女性圈层社区",
    COL["q16e"]: "韩系潮流实验室"
}

In [ ]:
for column in Q16_COLS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [ ]:
df["体验兴趣平均分"] = df[Q16_COLS].mean(axis=1)
df["高兴趣体验数量"] = df[Q16_COLS].ge(4).sum(axis=1)

In [ ]:
# 宽口径：至少一项评分4-5
df["至少一项高兴趣"] = (
    df["高兴趣体验数量"] >= 1
).astype(int)

In [ ]:
# 中等口径：至少三项评分4-5
df["至少三项高兴趣"] = (
    df["高兴趣体验数量"] >= 3
).astype(int)

In [ ]:
# 严格口径：五项平均分不低于4
df["整体升级高兴趣"] = (
    df["体验兴趣平均分"] >= 4
).astype(int)

=========================================================
8. 路径漏斗与断点指标
=========================================================

In [ ]:
n_total = len(df)
n_aware = int(df["品牌认知"].sum())
n_purchased = int(df["购买过3CE"].sum())
n_retained = int(df["经常购买3CE"].sum())

In [ ]:
n_unaware = n_total - n_aware
n_aware_not_buy = int(
    ((df["品牌认知"] == 1) & (df["购买过3CE"] == 0)).sum()
)
n_buy_not_retain = int(
    ((df["购买过3CE"] == 1) & (df["经常购买3CE"] == 0)).sum()
)

In [ ]:
brand_awareness_rate = n_aware / n_total
awareness_purchase_rate = (
    n_purchased / n_aware if n_aware else np.nan
)
purchase_retention_rate = (
    n_retained / n_purchased if n_purchased else np.nan
)

In [ ]:
awareness_break_rate = n_unaware / n_total
purchase_break_rate = (
    n_aware_not_buy / n_aware if n_aware else np.nan
)
retention_break_rate = (
    n_buy_not_retain / n_purchased if n_purchased else np.nan
)

In [ ]:
upgrade_broad_rate = df["至少一项高兴趣"].mean()
upgrade_medium_rate = df["至少三项高兴趣"].mean()
upgrade_strict_rate = df["整体升级高兴趣"].mean()

In [ ]:
funnel_summary = pd.DataFrame({
    "路径节点": [
        "总样本",
        "知道3CE",
        "购买过3CE",
        "经常购买3CE"
    ],
    "人数": [
        n_total,
        n_aware,
        n_purchased,
        n_retained
    ],
    "占总样本比例": [
        1,
        brand_awareness_rate,
        n_purchased / n_total,
        n_retained / n_total
    ]
})

In [ ]:
conversion_summary = pd.DataFrame({
    "路径指标": [
        "品牌认知率",
        "认知→购买转化率",
        "购买→经常购买率",
        "至少一项体验高兴趣率",
        "至少三项体验高兴趣率",
        "整体升级高兴趣率"
    ],
    "人数/分子": [
        n_aware,
        n_purchased,
        n_retained,
        int(df["至少一项高兴趣"].sum()),
        int(df["至少三项高兴趣"].sum()),
        int(df["整体升级高兴趣"].sum())
    ],
    "比例": [
        brand_awareness_rate,
        awareness_purchase_rate,
        purchase_retention_rate,
        upgrade_broad_rate,
        upgrade_medium_rate,
        upgrade_strict_rate
    ]
})

In [ ]:
breakpoint_summary = pd.DataFrame({
    "阶段": [
        "认知断点",
        "购买断点",
        "留存断点",
        "升级断点"
    ],
    "断点定义": [
        "完全不了解3CE",
        "知道3CE但没有购买",
        "购买过但没有形成经常购买",
        "五项体验平均分低于4"
    ],
    "断点人数": [
        n_unaware,
        n_aware_not_buy,
        n_buy_not_retain,
        int((df["整体升级高兴趣"] == 0).sum())
    ],
    "断点率": [
        awareness_break_rate,
        purchase_break_rate,
        retention_break_rate,
        1 - upgrade_strict_rate
    ],
    "说明": [
        "衡量品牌尚未覆盖的人群",
        "衡量认知未转化为首次购买的人群",
        "问卷缺少购买时间，因此为留存代理指标",
        "严格口径：五项体验平均分低于4"
    ]
})

In [ ]:
print("\n================ 路径转化结果 ================")

In [ ]:
for _, row in conversion_summary.iterrows():
    print(
        f'{row["路径指标"]}：'
        f'{row["比例"]:.1%}'
    )

In [ ]:
print("\n================ 路径断点结果 ================")

In [ ]:
for _, row in breakpoint_summary.iterrows():
    print(
        f'{row["阶段"]}：'
        f'{row["断点人数"]}人，'
        f'断点率{row["断点率"]:.1%}'
    )

=========================================================
9. 多选题分析函数
=========================================================

In [ ]:
def detect_separator(values):
    """
    检测问卷多选答案最可能使用的分隔符。
    不使用逗号作为分隔符，因为选项本身包含中文逗号。
    """
    text = " ".join(values.dropna().astype(str).head(100).tolist())

    candidates = [
        ("分号", r"[;；]"),
        ("竖线", r"[|｜]"),
        ("换行", r"[\r\n]+"),
        ("双竖线", r"\|\|")
    ]

    scores = {}

    for name, pattern in candidates:
        scores[name] = len(re.findall(pattern, text))

    best_name = max(scores, key=scores.get)

    if scores[best_name] == 0:
        return None

    separator_map = {
        "分号": r"[;；]",
        "竖线": r"[|｜]",
        "换行": r"[\r\n]+",
        "双竖线": r"\|\|"
    }

    return separator_map[best_name]

In [ ]:
def multi_select_rate(data, column, options=None):
    """
    多选题统计。

    优先使用固定选项进行整句匹配，避免把：
    “容易被种草，但买回来闲置”
    错误拆成两个选项。
    """
    valid = data[column].dropna().astype(str)

    denominator = len(valid)

    if denominator == 0:
        return pd.DataFrame(
            columns=["选项", "选择人数", "选择率"]
        )

    rows = []

    if options:
        for option in options:
            count = valid.str.contains(
                option,
                regex=False,
                na=False
            ).sum()

            rows.append({
                "选项": option,
                "选择人数": int(count),
                "选择率": count / denominator
            })

        result = pd.DataFrame(rows)

    else:
        separator = detect_separator(valid)

        if separator is None:
            values = valid.copy()
        else:
            values = (
                valid
                .str.split(separator, regex=True)
                .explode()
                .str.strip()
            )

        values = values[
            values.notna() &
            values.ne("")
        ]

        result = (
            values
            .value_counts()
            .rename_axis("选项")
            .reset_index(name="选择人数")
        )

        result["选择率"] = (
            result["选择人数"] / denominator
        )

    return (
        result
        .sort_values(
            ["选择人数", "选项"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

=========================================================
10. 固定选项设置
=========================================================

In [ ]:
platform_options = [
    "小红书",
    "抖音",
    "B站",
    "微博",
    "韩剧/短剧平台",
    "朋友推荐",
    "线下试妆"
]

In [ ]:
reason_options = [
    "表达自己的个性",
    "模仿喜欢的明星/影视角色",
    "符合特定场合的妆容需求（如演唱会）",
    "跟随潮流",
    "享受购买和使用过程",
    "其他"
]

In [ ]:
difficulty_options = [
    "色号太多，不知道怎么选择",
    "无法找准个人风格定位",
    "容易被种草，但买回来闲置",
    "其他"
]

In [ ]:
q10a_options = [
    "产品效果符合预期",
    "色彩和妆效适合我",
    "包装与视觉设计吸引我",
    "能代表我喜欢的韩系风格",
    "新品更新有吸引力",
    "明星/KOL或社交媒体影响",
    "性价比较高",
    "已经形成使用习惯",
    "品牌活动或会员权益",
    "其他"
]

In [ ]:
q11a_options = [
    "更懂我的个性化产品推荐",
    "根据不同情景需求推荐完整妆容",
    "新品试用或专属优惠",
    "更丰富的色号和产品类型",
    "线上虚拟试妆",
    "与喜欢的明星/KOL联名",
    "用户共创、投票或限定活动",
    "更持续的品牌互动与陪伴"
]

In [ ]:
q10b_options = [
    "产品体验没有达到预期",
    "产品不错，但没有持续关注品牌",
    "不清楚新品或色号是否适合自己",
    "产品或品牌风格不再适合自己",
    "感觉3CE只是阶段性流行",
    "与其他品牌相比缺少新鲜感",
    "其他品牌更懂我的需求",
    "其他品牌更有互动感或陪伴感",
    "价格或性价比因素",
    "购买渠道不方便",
    "其他"
]

In [ ]:
q11b_options = [
    "根据我的特征推荐色号",
    "根据生活情景需求提供妆容方案",
    "虚拟试妆或AI妆容顾问",
    "更有吸引力的新品和限定系列",
    "会员权益、复购优惠",
    "与用户共同设计新品",
    "持续记录和更新个人风格档案",
    "明星/KOL合作",
    "暂时没有什么能让我重新购买"
]

In [ ]:
q10c_options = [
    "不确定产品是否适合自己",
    "不知道应该选择哪个色号",
    "对产品效果缺乏了解",
    "品牌风格不是我的类型",
    "其他品牌能够满足我的需求",
    "价格或性价比不合适",
    "缺少试用或试妆机会",
    "没有产生实际购买需求",
    "购买渠道不方便",
    "其他"
]

In [ ]:
q11c_options = [
    "AI分析适合我的色号和风格",
    "在线虚拟试妆",
    "给予不同生活情景下的完整妆容方案指导",
    "新用户试用装或首次购买优惠",
    "真实用户的妆效展示",
    "明星/KOL推荐",
    "朋友推荐或社交分享",
    "校园限定或城市限定活动",
    "参与投票、共创新品",
    "都不会明显提高"
]

In [ ]:
new_brand_options = [
    "独特的色彩与视觉设计",
    "能感知产品在自己脸上的效果",
    "清晰的品牌风格",
    "品牌给予个性化推荐",
    "品牌给予不同生活情境下的妆容方案指导",
    "虚拟试妆等互动体验",
    "明星/KOL合作",
    "社交媒体热门内容",
    "朋友推荐",
    "新用户优惠或试用",
    "有参与感的活动或共创",
    "其他"
]

=========================================================
11. 总体消费者痛点
=========================================================

In [ ]:
platform_summary = multi_select_rate(
    df,
    COL["platform"],
    platform_options
)

In [ ]:
purchase_reason_summary = multi_select_rate(
    df,
    COL["reason"],
    reason_options
)

In [ ]:
difficulty_summary = multi_select_rate(
    df,
    COL["difficulty"],
    difficulty_options
)

=========================================================
12. 各路径阶段的分支题分析
=========================================================

In [ ]:
def analyze_branch(data, column_key, options):
    column = COL[column_key]

    if column not in data.columns:
        return pd.DataFrame(
            columns=["选项", "选择人数", "选择率"]
        )

    return multi_select_rate(
        data,
        column,
        options
    )

In [ ]:
# 经常购买者：留存驱动
q10a_summary = analyze_branch(
    df,
    "q10a",
    q10a_options
)

In [ ]:
# 经常购买者：提高购买频率的方法
q11a_summary = analyze_branch(
    df,
    "q11a",
    q11a_options
)

In [ ]:
# 买过1-2次：未继续购买原因，即留存断点
q10b_summary = analyze_branch(
    df,
    "q10b",
    q10b_options
)

In [ ]:
# 买过1-2次：重新激活方式
q11b_summary = analyze_branch(
    df,
    "q11b",
    q11b_options
)

In [ ]:
# 听说但未购买：首次购买断点
q10c_summary = analyze_branch(
    df,
    "q10c",
    q10c_options
)

In [ ]:
# 听说但未购买：首次购买促进方式
q11c_summary = analyze_branch(
    df,
    "q11c",
    q11c_options
)

In [ ]:
# 完全不了解：新品牌关注驱动
new_brand_summary = analyze_branch(
    df,
    "new_brand_reason",
    new_brand_options
)

=========================================================
13. Q7购买不适配问题
=========================================================

In [ ]:
mismatch_order = [
    "经常",
    "偶尔",
    "很少",
    "从没有"
]

In [ ]:
mismatch_summary = (
    df[COL["mismatch"]]
    .value_counts(dropna=False)
    .reindex(mismatch_order)
    .fillna(0)
    .astype(int)
    .rename_axis("频率")
    .reset_index(name="人数")
)

In [ ]:
mismatch_summary["比例"] = (
    mismatch_summary["人数"] /
    mismatch_summary["人数"].sum()
)

In [ ]:
df["经常或偶尔买后不适合"] = (
    df[COL["mismatch"]]
    .isin(["经常", "偶尔"])
    .astype(int)
)

In [ ]:
mismatch_by_stage = pd.crosstab(
    df["消费者阶段"],
    df["经常或偶尔买后不适合"],
    normalize="index"
).reset_index()

In [ ]:
mismatch_by_stage = mismatch_by_stage.rename(
    columns={
        0: "很少或从没有",
        1: "经常或偶尔"
    }
)

=========================================================
14. Q16体验评分
=========================================================

In [ ]:
q16_summary = pd.DataFrame({
    "体验": [
        Q16_NAMES[column]
        for column in Q16_COLS
    ],
    "平均分": [
        df[column].mean()
        for column in Q16_COLS
    ],
    "中位数": [
        df[column].median()
        for column in Q16_COLS
    ],
    "4-5分人数": [
        int(df[column].ge(4).sum())
        for column in Q16_COLS
    ],
    "4-5分比例": [
        df[column].ge(4).mean()
        for column in Q16_COLS
    ],
    "1-2分比例": [
        df[column].le(2).mean()
        for column in Q16_COLS
    ],
    "有效样本": [
        int(df[column].count())
        for column in Q16_COLS
    ]
}).sort_values(
    "4-5分比例",
    ascending=False
).reset_index(drop=True)

In [ ]:
def cronbach_alpha(items):
    clean = items.dropna()

    if clean.shape[0] < 2 or clean.shape[1] < 2:
        return np.nan

    k = clean.shape[1]
    item_variance = clean.var(axis=0, ddof=1).sum()
    total_variance = clean.sum(axis=1).var(ddof=1)

    if total_variance == 0:
        return np.nan

    return (
        k / (k - 1) *
        (1 - item_variance / total_variance)
    )

In [ ]:
alpha = cronbach_alpha(df[Q16_COLS])

In [ ]:
print(f"\nQ16 Cronbach's Alpha：{alpha:.3f}")

In [ ]:
# 按消费者阶段比较Q16平均分
q16_by_stage = (
    df.groupby("消费者阶段")[Q16_COLS]
    .mean()
    .rename(columns=Q16_NAMES)
    .reset_index()
)

=========================================================
15. 卡方检验与效应量
=========================================================

In [ ]:
def chi_square_test(data, x, y):
    clean = data[[x, y]].dropna()
    table = pd.crosstab(clean[x], clean[y])

    if table.shape[0] < 2 or table.shape[1] < 2:
        return {
            "变量": x,
            "结果变量": y,
            "卡方值": np.nan,
            "p值": np.nan,
            "Cramers_V": np.nan,
            "样本量": len(clean),
            "结论": "类别不足，无法检验"
        }

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    rows, columns = table.shape

    denominator = n * min(
        rows - 1,
        columns - 1
    )

    cramers_v = (
        np.sqrt(chi2 / denominator)
        if denominator > 0
        else np.nan
    )

    if p_value < 0.05:
        conclusion = "存在显著关联"
    else:
        conclusion = "未发现显著关联"

    return {
        "变量": x,
        "结果变量": y,
        "卡方值": chi2,
        "p值": p_value,
        "Cramers_V": cramers_v,
        "样本量": n,
        "结论": conclusion
    }

In [ ]:
chi_results = []

In [ ]:
test_variables = [
    COL["age"],
    COL["city"],
    COL["kdrama"],
    COL["role_identification"]
]

In [ ]:
for variable in test_variables:
    if variable in df.columns:
        chi_results.append(
            chi_square_test(
                df,
                variable,
                "购买过3CE"
            )
        )

        chi_results.append(
            chi_square_test(
                df,
                variable,
                "经常购买3CE"
            )
        )

        chi_results.append(
            chi_square_test(
                df,
                variable,
                "整体升级高兴趣"
            )
        )

In [ ]:
chi_square_summary = pd.DataFrame(chi_results)

=========================================================
16. 购买影响因素Logistic回归
=========================================================

In [ ]:
regression_result = pd.DataFrame()

In [ ]:
model_columns = [
    "购买过3CE",
    COL["age"],
    COL["city"],
    COL["kdrama"],
    "经常或偶尔买后不适合",
    "体验兴趣平均分"
]

In [ ]:
model_df = df[model_columns].dropna().copy()

In [ ]:
try:
    purchase_model = smf.logit(
        formula=(
            f'Q("购买过3CE")'
            f' ~ C(Q("{COL["age"]}"))'
            f' + C(Q("{COL["city"]}"))'
            f' + C(Q("{COL["kdrama"]}"))'
            f' + Q("经常或偶尔买后不适合")'
            f' + Q("体验兴趣平均分")'
        ),
        data=model_df
    ).fit(disp=False)

    confidence_interval = purchase_model.conf_int()

    regression_result = pd.DataFrame({
        "变量": purchase_model.params.index,
        "OR": np.exp(purchase_model.params.values),
        "95%CI下限": np.exp(
            confidence_interval[0].values
        ),
        "95%CI上限": np.exp(
            confidence_interval[1].values
        ),
        "p值": purchase_model.pvalues.values
    })

    regression_result["显著性"] = np.where(
        regression_result["p值"] < 0.05,
        "显著",
        "不显著"
    )

In [ ]:
except Exception as error:
    print("\n购买回归未成功：", error)

    regression_result = pd.DataFrame({
        "说明": [
            f"模型未成功运行：{error}"
        ]
    })

=========================================================
17. 自动提取各阶段Top 3断点与机会
=========================================================

In [ ]:
def get_top_items(summary, n=3):
    if summary.empty:
        return []

    valid = summary[
        summary["选择人数"] > 0
    ].head(n)

    return [
        (
            row["选项"],
            int(row["选择人数"]),
            float(row["选择率"])
        )
        for _, row in valid.iterrows()
    ]

In [ ]:
def format_top_items(items):
    if not items:
        return "暂无有效数据"

    return "；".join([
        f"{name}（{rate:.1%}，{count}人）"
        for name, count, rate in items
    ])

In [ ]:
cognition_top = get_top_items(
    new_brand_summary,
    3
)

In [ ]:
purchase_top = get_top_items(
    q10c_summary,
    3
)

In [ ]:
retention_top = get_top_items(
    q10b_summary,
    3
)

In [ ]:
upgrade_top = [
    (
        row["体验"],
        int(row["4-5分人数"]),
        float(row["4-5分比例"])
    )
    for _, row in q16_summary.head(3).iterrows()
]

In [ ]:
research_support = pd.DataFrame({
    "路径阶段": [
        "认知",
        "购买",
        "留存",
        "升级"
    ],
    "量化断点": [
        (
            f"未认知人数{n_unaware}人，"
            f"认知断点率{awareness_break_rate:.1%}"
        ),
        (
            f"知道但未购买{n_aware_not_buy}人，"
            f"购买断点率{purchase_break_rate:.1%}"
        ),
        (
            f"购买但未形成经常购买{n_buy_not_retain}人，"
            f"留存断点率{retention_break_rate:.1%}"
        ),
        (
            f"整体升级高兴趣率{upgrade_strict_rate:.1%}；"
            f"升级兴趣不足率{1-upgrade_strict_rate:.1%}"
        )
    ],
    "主要调研证据": [
        format_top_items(cognition_top),
        format_top_items(purchase_top),
        format_top_items(retention_top),
        format_top_items(upgrade_top)
    ],
    "建议解释": [
        (
            "识别完全不了解人群最容易被什么品牌内容触达，"
            "用于选择认知入口和传播内容。"
        ),
        (
            "重点判断适配不确定、选色困难、效果信息不足、"
            "缺少试妆等因素是否阻碍首次购买。"
        ),
        (
            "重点判断产品体验、持续关注、风格适配、新鲜感、"
            "性价比和品牌互动是否阻碍复购。"
        ),
        (
            "将场景助手、AI风格探索、韩系潮流内容作为升级体验，"
            "按照4-5分比例决定产品优先级。"
        )
    ]
})

=========================================================
18. 输出控制台结论
=========================================================

In [ ]:
print("\n================ 核心调研支撑 ================")

In [ ]:
for _, row in research_support.iterrows():
    print(f'\n【{row["路径阶段"]}阶段】')
    print("量化断点：", row["量化断点"])
    print("主要证据：", row["主要调研证据"])
    print("解释建议：", row["建议解释"])

In [ ]:
print("\n================ 总体购买困难Top 3 ================")

In [ ]:
print(
    difficulty_summary[
        ["选项", "选择人数", "选择率"]
    ].head(3).to_string(index=False)
)

In [ ]:
print("\n================ Q16体验优先级 ================")

In [ ]:
print(
    q16_summary[
        ["体验", "平均分", "4-5分比例", "1-2分比例"]
    ].to_string(index=False)
)

=========================================================
19. 绘制路径漏斗图
=========================================================

In [ ]:
stage_color = [
    "#D9D9D9",
    "#B9A7E8",
    "#8D6FD1",
    "#5D3E9F"
]

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
bars = plt.bar(
    funnel_summary["路径节点"],
    funnel_summary["人数"],
    color=stage_color
)

In [ ]:
plt.title(
    "3CE消费者路径漏斗：认知—购买—经常购买",
    fontsize=15,
    pad=15
)

In [ ]:
plt.ylabel("人数")
plt.xlabel("")

In [ ]:
for bar, number in zip(
    bars,
    funnel_summary["人数"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(n_total * 0.015, 1),
        f"{number}人\n{number/n_total:.1%}",
        ha="center",
        va="bottom",
        fontsize=10
    )

In [ ]:
plt.ylim(
    0,
    max(funnel_summary["人数"]) * 1.18
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR / "01_消费者路径漏斗.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.close()

=========================================================
20. 绘制断点率图
=========================================================

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
bars = plt.bar(
    breakpoint_summary["阶段"],
    breakpoint_summary["断点率"],
    color=[
        "#A6A6A6",
        "#E9B949",
        "#D96C6C",
        "#6FA8DC"
    ]
)

In [ ]:
plt.title(
    "3CE消费者路径各阶段断点率",
    fontsize=15,
    pad=15
)

In [ ]:
plt.ylabel("断点率")
plt.xlabel("")
plt.ylim(0, 1)

In [ ]:
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda value, position: f"{value:.0%}"
    )
)

In [ ]:
for bar, rate in zip(
    bars,
    breakpoint_summary["断点率"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.025,
        f"{rate:.1%}",
        ha="center",
        va="bottom"
    )

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR / "02_路径断点率.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.close()

=========================================================
21. 绘制购买困难图
=========================================================

In [ ]:
difficulty_plot = (
    difficulty_summary[
        difficulty_summary["选择人数"] > 0
    ]
    .sort_values("选择率", ascending=True)
)

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
bars = plt.barh(
    difficulty_plot["选项"],
    difficulty_plot["选择率"],
    color="#8D6FD1"
)

In [ ]:
plt.title(
    "消费者购买彩妆的主要困难",
    fontsize=15,
    pad=15
)

In [ ]:
plt.xlabel("选择率")
plt.ylabel("")

In [ ]:
plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda value, position: f"{value:.0%}"
    )
)

In [ ]:
for bar, rate in zip(
    bars,
    difficulty_plot["选择率"]
):
    plt.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1%}",
        va="center"
    )

In [ ]:
plt.xlim(
    0,
    max(difficulty_plot["选择率"].max() * 1.25, 0.1)
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR / "03_购买彩妆主要困难.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.close()

=========================================================
22. 绘制Q16体验优先级
=========================================================

In [ ]:
q16_plot = q16_summary.sort_values(
    "4-5分比例",
    ascending=True
)

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
bars = plt.barh(
    q16_plot["体验"],
    q16_plot["4-5分比例"],
    color="#D96C9D"
)

In [ ]:
plt.title(
    "3CE升级体验优先级：评分4—5分比例",
    fontsize=15,
    pad=15
)

In [ ]:
plt.xlabel("高兴趣比例")
plt.ylabel("")

In [ ]:
plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda value, position: f"{value:.0%}"
    )
)

In [ ]:
for bar, rate in zip(
    bars,
    q16_plot["4-5分比例"]
):
    plt.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1%}",
        va="center"
    )

In [ ]:
plt.xlim(
    0,
    min(
        max(q16_plot["4-5分比例"].max() * 1.2, 0.1),
        1
    )
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR / "04_升级体验优先级.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.close()

=========================================================
23. 导出Excel
=========================================================

In [ ]:
with pd.ExcelWriter(
    EXCEL_OUTPUT,
    engine="openpyxl"
) as writer:

    funnel_summary.to_excel(
        writer,
        sheet_name="01路径漏斗",
        index=False
    )

    conversion_summary.to_excel(
        writer,
        sheet_name="02转化率",
        index=False
    )

    breakpoint_summary.to_excel(
        writer,
        sheet_name="03断点率",
        index=False
    )

    research_support.to_excel(
        writer,
        sheet_name="04调研支撑总结",
        index=False
    )

    difficulty_summary.to_excel(
        writer,
        sheet_name="05总体购买困难",
        index=False
    )

    purchase_reason_summary.to_excel(
        writer,
        sheet_name="06购买原因",
        index=False
    )

    platform_summary.to_excel(
        writer,
        sheet_name="07内容平台",
        index=False
    )

    q10a_summary.to_excel(
        writer,
        sheet_name="08留存驱动",
        index=False
    )

    q11a_summary.to_excel(
        writer,
        sheet_name="09提高购买频率",
        index=False
    )

    q10b_summary.to_excel(
        writer,
        sheet_name="10留存断点",
        index=False
    )

    q11b_summary.to_excel(
        writer,
        sheet_name="11重新激活",
        index=False
    )

    q10c_summary.to_excel(
        writer,
        sheet_name="12首购断点",
        index=False
    )

    q11c_summary.to_excel(
        writer,
        sheet_name="13首购促进",
        index=False
    )

    new_brand_summary.to_excel(
        writer,
        sheet_name="14认知驱动",
        index=False
    )

    mismatch_summary.to_excel(
        writer,
        sheet_name="15买后不适配",
        index=False
    )

    mismatch_by_stage.to_excel(
        writer,
        sheet_name="16各阶段不适配",
        index=False
    )

    q16_summary.to_excel(
        writer,
        sheet_name="17升级体验",
        index=False
    )

    q16_by_stage.to_excel(
        writer,
        sheet_name="18分阶段体验评分",
        index=False
    )

    chi_square_summary.to_excel(
        writer,
        sheet_name="19显著性检验",
        index=False
    )

    regression_result.to_excel(
        writer,
        sheet_name="20购买回归",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="21清洗后原始数据",
        index=False
    )

=========================================================
24. 简单美化Excel
=========================================================

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

In [ ]:
workbook = load_workbook(EXCEL_OUTPUT)

In [ ]:
header_fill = PatternFill(
    "solid",
    fgColor="5D3E9F"
)

In [ ]:
header_font = Font(
    color="FFFFFF",
    bold=True
)

In [ ]:
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions

    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    for column_cells in worksheet.columns:
        column_letter = column_cells[0].column_letter

        max_length = 0

        for cell in column_cells:
            value = "" if cell.value is None else str(cell.value)
            max_length = max(
                max_length,
                len(value)
            )

        worksheet.column_dimensions[column_letter].width = min(
            max(max_length + 2, 10),
            45
        )

    for row in worksheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    # 比例列显示为百分数
    for cell in worksheet[1]:
        if cell.value and any(
            keyword in str(cell.value)
            for keyword in ["比例", "断点率"]
        ):
            column_index = cell.column

            for row_number in range(
                2,
                worksheet.max_row + 1
            ):
                worksheet.cell(
                    row=row_number,
                    column=column_index
                ).number_format = "0.0%"

In [ ]:
workbook.save(EXCEL_OUTPUT)

=========================================================
25. 完成提示
=========================================================

In [ ]:
print("\n================ 分析完成 ================")
print(f"Excel分析结果：{EXCEL_OUTPUT}")
print(f"图表保存目录：{OUTPUT_DIR}")

In [ ]:
print("\n生成的主要文件：")
print("1. 3CE消费者路径断点分析.xlsx")
print("2. 01_消费者路径漏斗.png")
print("3. 02_路径断点率.png")
print("4. 03_购买彩妆主要困难.png")
print("5. 04_升级体验优先级.png")

In [ ]:
print(
    "\n提示：购买→经常购买率是留存代理指标，"
    "因为本问卷没有记录复购次数和最近购买时间。"
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

In [ ]:
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
names = [
    "场景妆容助手",
    "韩系潮流实验室",
    "AI个人风格探索"
]

In [ ]:
values = [70.1, 67.1, 66.7]

In [ ]:
colors = ["#6741B5", "#8061C5", "#9A82D4"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

In [ ]:
bars = ax.barh(
    names,
    values,
    color=colors,
    height=0.55
)

In [ ]:
ax.invert_yaxis()
ax.set_xlim(0, 80)
ax.xaxis.set_major_formatter(PercentFormatter(100))

In [ ]:
ax.set_title(
    "3CE升级体验兴趣对比",
    fontsize=17,
    fontweight="bold",
    pad=18
)

In [ ]:
ax.set_xlabel("评分4–5分的受访者比例", fontsize=11)
ax.set_ylabel("")

In [ ]:
for bar, value in zip(bars, values):
    ax.text(
        value + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=12,
        fontweight="bold",
        color="#4B267E"
    )

In [ ]:
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

In [ ]:
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    "3CE升级体验兴趣对比.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

In [ ]:
plt.show()